In [ ]:
from project_config import CATALOG_ROOT, OUTPUT_ROOT, EVALUATION_SPLIT, HAILRU_CHECKPOINT, load_pickle, safe_divide
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import numpy as np
import pandas as pd
from tqdm import tqdm


In [ ]:
setname = EVALUATION_SPLIT
data_dic = load_pickle(CATALOG_ROOT / setname / 'data_dic_nwp.pkl')
mask_dic = load_pickle(CATALOG_ROOT / setname / 'mask_dic_all.pkl')
train_dic_matrix = load_pickle(CATALOG_ROOT / setname / 'train_dic_matrix.pkl')


In [ ]:
class SpatialAttention(nn.Module):

    def __init__(self):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, 7, 1, 3)
        self.act = nn.Sigmoid()

    def forward(self, x):
        maxpool = torch.max(x, dim=1, keepdim=True)[0]
        avgpool = torch.mean(x, dim=1, keepdim=True)
        SA = self.act(self.conv(torch.cat((maxpool, avgpool), dim=1)))
        return SA * x

class ChannelAttention(nn.Module):

    def __init__(self, dim):
        super(ChannelAttention, self).__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.maxpool = nn.AdaptiveMaxPool2d(1)
        self.conv_shared = nn.Sequential(nn.Conv2d(dim, dim // 16, 1, 1), nn.LeakyReLU(), nn.Conv2d(dim // 16, dim, 1, 1))
        self.act = nn.Sigmoid()

    def forward(self, x):
        maxpool = self.conv_shared(self.maxpool(x))
        avgpool = self.conv_shared(self.avgpool(x))
        CA = self.act(maxpool + avgpool)
        return CA * x

class CBAM(nn.Module):

    def __init__(self, dim):
        super(CBAM, self).__init__()
        self.CA = ChannelAttention(dim)
        self.SA = SpatialAttention()

    def forward(self, x):
        x = self.CA(x)
        x = self.SA(x)
        return x

class IRCBAM(nn.Module):

    def __init__(self, inchannels):
        super(IRCBAM, self).__init__()
        self.c = inchannels
        self.act = nn.LeakyReLU()
        self.branch1 = nn.Sequential(nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.branch2 = nn.Sequential(nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act, nn.Conv2d(self.c // 4, self.c // 4, 3, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.branch3 = nn.Sequential(nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act, nn.Conv2d(self.c // 4, self.c // 4, 3, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act, nn.Conv2d(self.c // 4, self.c // 4, 3, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.branch4 = nn.Sequential(nn.AvgPool2d(3, 1, 1), nn.Conv2d(self.c, self.c // 4, 1, 1, 'same'), nn.BatchNorm2d(self.c // 4), self.act)
        self.CBAM = CBAM(inchannels)

    def forward(self, x):
        identity = x
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        cat = torch.cat([b1, b2, b3, b4], dim=1)
        cat = self.CBAM(cat)
        output = 0.3 * cat + identity
        return output

class HRUnet(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(HRUnet, self).__init__()
        chans = [64, 256, 1024]
        ks = 3
        self.act = nn.LeakyReLU(inplace=True)
        self.act_output = nn.Sigmoid()
        self.start_conv = nn.Sequential(nn.Conv2d(in_channels, chans[0], 2, 1, 0), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_11 = IRCBAM(chans[0])
        self.ResBlock_12 = IRCBAM(chans[0])
        self.DownConv_1 = nn.PixelUnshuffle(2)
        self.ResBlock_21 = IRCBAM(chans[1])
        self.ResBlock_22 = IRCBAM(chans[1])
        self.DownConv_2 = nn.PixelUnshuffle(2)
        self.ResBlock_31 = IRCBAM(chans[2])
        self.ResBlock_32 = IRCBAM(chans[2])
        self.ResBlock_33 = IRCBAM(chans[2])
        self.ResBlock_34 = IRCBAM(chans[2])
        self.UpConv_1 = nn.PixelShuffle(2)
        self.UpConv_next_1 = nn.Sequential(nn.Conv2d(chans[1] * 2, chans[1], ks, 1, 1), nn.BatchNorm2d(chans[1]), self.act)
        self.ResBlock_23 = IRCBAM(chans[1])
        self.ResBlock_24 = IRCBAM(chans[1])
        self.UpConv_2 = nn.PixelShuffle(2)
        self.UpConv_next_2 = nn.Sequential(nn.Conv2d(chans[0] * 2, chans[0], ks, 1, 1), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_13 = IRCBAM(chans[0])
        self.ResBlock_14 = IRCBAM(chans[0])
        self.final_conv = nn.Sequential(nn.Conv2d(chans[0], out_channels, 2, 1, 1), self.act_output)

    def forward(self, x, mask):
        x = self.start_conv(x)
        x = self.ResBlock_11(x)
        x = self.ResBlock_12(x)
        x_pass_1 = x
        x = self.DownConv_1(x)
        x = self.ResBlock_21(x)
        x = self.ResBlock_22(x)
        x_pass_2 = x
        x = self.DownConv_2(x)
        x = self.ResBlock_31(x)
        x = self.ResBlock_32(x)
        x = self.ResBlock_33(x)
        x = self.ResBlock_34(x)
        x = self.UpConv_1(x)
        x = torch.cat((x, x_pass_2), dim=1)
        x = self.UpConv_next_1(x)
        x = self.ResBlock_23(x)
        x = self.ResBlock_24(x)
        x = self.UpConv_2(x)
        x = torch.cat((x, x_pass_1), dim=1)
        x = self.UpConv_next_2(x)
        x = self.ResBlock_13(x)
        x = self.ResBlock_14(x)
        x = self.final_conv(x)
        x = x * mask
        return x
in_channels = 102
out_channels = 20
model = HRUnet(in_channels, out_channels)

class Loss(nn.Module):

    def __init__(self, alpha, beta):
        super(Loss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        if torch.sum(y_true) > 0:
            temp = torch.square(y_pred - y_true) * ((y_true + self.alpha) / (1 + self.alpha))
        else:
            temp = self.beta * torch.square(y_pred - y_true) * ((y_true + self.alpha) / (1 + self.alpha))
        return torch.sum(temp) / (temp.size()[1] * temp.size()[2] * temp.size()[3])

class LightningModel(pl.LightningModule):

    def __init__(self, alpha, beta):
        super().__init__()
        self.model = HRUnet(in_channels, out_channels)
        self.criterion = Loss(alpha, beta)

    def forward(self, x, mask):
        return self.model(x, mask)

    def training_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        loss = self.criterion(outputs, labels)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        val_loss = self.criterion(outputs, labels)
        self.log('val_loss', val_loss)
        return val_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.0001)
model = LightningModel(0.1, 1)
CKPT_PATH = HAILRU_CHECKPOINT
model = LightningModel.load_from_checkpoint(CKPT_PATH, map_location='cpu', alpha=0.1, beta=1)
model.eval()
model.to('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
TP_t = torch.tensor(0, device=model.device, dtype=torch.int64)
TN_t = torch.tensor(0, device=model.device, dtype=torch.int64)
FP_t = torch.tensor(0, device=model.device, dtype=torch.int64)
FN_t = torch.tensor(0, device=model.device, dtype=torch.int64)
thre = 0.48
width = 1
kernel_size = 2 * width + 1
for case_id in tqdm(train_dic_matrix, total=len(train_dic_matrix), desc='Processing Cases'):
    coords = np.array(list(mask_dic[case_id].values()))
    x_coords = torch.tensor(coords[:, 0], dtype=torch.long, device=model.device)
    y_coords = torch.tensor(coords[:, 1], dtype=torch.long, device=model.device)
    for index in train_dic_matrix[case_id]:
        input_data = np.load(train_dic_matrix[case_id][index]['input'])['arr_0']
        mask_data = np.load(train_dic_matrix[case_id][index]['mask'])['arr_0']
        output_data = np.load(train_dic_matrix[case_id][index]['output'])['arr_0']
        input_tensor = torch.tensor(input_data, dtype=torch.float32, device=model.device).unsqueeze(0)
        mask_tensor = torch.tensor(mask_data, dtype=torch.float32, device=model.device).unsqueeze(0).unsqueeze(0)
        output_tensor = torch.tensor(output_data, dtype=torch.float32, device=model.device).unsqueeze(0)
        with torch.no_grad():
            output_predict = model(input_tensor, mask_tensor)
            pooled_predict = F.max_pool2d(output_predict, kernel_size=kernel_size, stride=1, padding=width)
            pred_vals = pooled_predict[0, :, y_coords, x_coords]
            true_vals = output_tensor[0, :, y_coords, x_coords]
            pred_pos = pred_vals >= thre
            true_pos = true_vals >= 0.99
            TP_t += (pred_pos & true_pos).sum()
            FP_t += (pred_pos & ~true_pos).sum()
            FN_t += (~pred_pos & true_pos).sum()
            TN_t += (~pred_pos & ~true_pos).sum()
TP = TP_t.item()
TN = TN_t.item()
FP = FP_t.item()
FN = FN_t.item()
print('model:', CKPT_PATH)
print('TP', TP, 'TN', TN, 'FP', FP, 'FN', FN)
accuracy = safe_divide(TP + TN, TP + TN + FP + FN)
precision = safe_divide(TP, TP + FP)
recall = safe_divide(TP, TP + FN)
denominator = TP + FP + FN
ts = safe_divide(TP, denominator)
r = safe_divide((TP + FP) * (TP + FN), TP + TN + FP + FN)
ets = safe_divide(TP - r, denominator - r)
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'TS Score:  {ts:.4f}')
print(f'ETS Score:  {ets:.4f}')
pd.DataFrame([dict(TP=TP, TN=TN, FP=FP, FN=FN, Accuracy=accuracy, Precision=precision, Recall=recall, TS=ts, ETS=ets)]).to_csv(OUTPUT_ROOT / 'overall_metrics.csv', index=False)
